# Fine-tune Parakeet for Speech Recognition

In this notebook, we fine-tune [NVIDIA Parakeet CTC](https://huggingface.co/nvidia/parakeet-ctc-1.1b) for automatic speech recognition. We cover both full fine-tuning and LoRA-based parameter-efficient training.

Parakeet is NVIDIA's CTC-based speech recognition model. Fine-tuning on domain-specific data can improve transcription accuracy for your use case.

**What we'll cover:**
- Loading and preprocessing an ASR dataset
- Setting up the Parakeet CTC data collator
- Full fine-tuning
- LoRA fine-tuning
- Running inference

In [1]:
!pip install -q transformers datasets accelerate peft librosa soundfile jiwer

## Load Dataset

We use a small LibriSpeech subset for demonstration. Replace with your own dataset for real training — just ensure it has `audio` (16kHz) and `text` columns.

In [2]:
from datasets import load_dataset, Audio

dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

train_dataset = dataset
eval_dataset = dataset

print(f"Samples: {len(dataset)}")
print(dataset[0].keys())

Samples: 73


dict_keys(['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'])


## Load Model and Processor

In [3]:
import torch
from transformers import AutoModelForCTC, AutoProcessor

model_checkpoint = "nvidia/parakeet-ctc-1.1b"

processor = AutoProcessor.from_pretrained(model_checkpoint)
model = AutoModelForCTC.from_pretrained(model_checkpoint)

print(f"Model: {model_checkpoint}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py


Model: nvidia/parakeet-ctc-1.1b
Parameters: 1062.5M


## Data Collator

The data collator processes raw audio and text through the Parakeet processor, which handles feature extraction and label tokenization for CTC training.

In [4]:
class ParakeetDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        texts = [f["text"] for f in features]
        audios = [f["audio"]["array"] for f in features]

        inputs = self.processor(
            audio=audios,
            text=texts,
            sampling_rate=self.processor.feature_extractor.sampling_rate,
        )
        return inputs

data_collator = ParakeetDataCollator(processor)

## Full Fine-tuning

In [5]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./parakeet-finetuned",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    max_steps=50,  # increase for real training
    bf16=True,
    logging_steps=10,
    eval_steps=50,
    save_steps=50,
    eval_strategy="steps",
    save_strategy="steps",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss,Validation Loss
50,0.769800,0.338426


TrainOutput(global_step=50, training_loss=0.9250845718383789, metrics={'train_runtime': 76.1967, 'train_samples_per_second': 5.25, 'train_steps_per_second': 0.656, 'total_flos': 1.6125944457450576e+17, 'train_loss': 0.9250845718383789, 'epoch': 5.0})

## LoRA Fine-tuning

Apply LoRA to the model's linear layers for parameter-efficient training. This is useful when GPU memory is limited.

In [6]:
from peft import LoraConfig, get_peft_model

model_lora = AutoModelForCTC.from_pretrained(model_checkpoint)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["linear"],
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()

trainable params: 57,344 || all params: 1,062,597,633 || trainable%: 0.0054


In [7]:
training_args_lora = TrainingArguments(
    output_dir="./parakeet-finetuned-lora",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    max_steps=50,  # increase for real training
    bf16=True,
    logging_steps=10,
    eval_steps=50,
    save_steps=50,
    eval_strategy="steps",
    save_strategy="steps",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=1,
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

trainer_lora.train()

Step,Training Loss,Validation Loss
50,2.271600,0.618370


TrainOutput(global_step=50, training_loss=1.7661978340148925, metrics={'train_runtime': 46.6411, 'train_samples_per_second': 8.576, 'train_steps_per_second': 1.072, 'total_flos': 1.6126814754952272e+17, 'train_loss': 1.7661978340148925, 'epoch': 5.0})

## Inference

In [8]:
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    feature_extractor=processor.feature_extractor,
    tokenizer=processor.tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

test_ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation[:3]")
test_ds = test_ds.cast_column("audio", Audio(sampling_rate=16000))

for i, sample in enumerate(test_ds):
    result = pipe(sample["audio"]["array"])
    print(f"Sample {i+1}:")
    print(f"  Ground truth: {sample['text']}")
    print(f"  Prediction:   {result['text']}")
    print()

Device set to use cuda:0


Sample 1:
  Ground truth: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
  Prediction:   mister quilter is the apostle of the middle class and we are glad to welcome his gospel

Sample 2:
  Ground truth: NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER
  Prediction:   nor is mister quilter's manner less interesting than his matter

Sample 3:
  Ground truth: HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND
  Prediction:   he tells us that at this festive season of the year with christmas and roast beef looming before us similes drawn from eating and its results occur most readily to the mind

